### Full workflow for XRD-conditioned crystal structure recovery (CHILI-100K Study)

In [ ]:
import __init__
import pandas as pd
from _utils._notebook_utils.x_xrd_chili100k_utils import get_stratified_metrics_xrd

### Two-Pass Finetuning Setup

The model is finetuned in two sequential passes before evaluation:

- **Pass 1** - finetune on a broad XRD-labelled dataset (Mattergen/Alex-MP-20) to teach the model the XRD condition format and get initial exposure to diverse crystal chemistries
- **Pass 2** - continue finetuning on CHILI-100K with the same XRD conditioning, using the leakproof stratified split built below

The text-only baseline mirrors this exact pipeline but with XRD info stripped out at every stage, so any metric difference is attributable to the XRD signal alone.

#### 1st pass finetune - Mattergen XRD
- Dataset Source: [Mattergen Alex-MP-20](https://github.com/microsoft/mattergen/tree/main/data-release/alex-mp)
  - Columns: Database (manual) 
  - Reduced Formula (Source)
  - CIF (pmg - CifWriter with symprec 0.1)
  - XRD 'Condition Vector' (with [_calculate_theor_XRD.py](_utils/_preprocessing/_calculate_theor_XRD.py))
    - pmg - XRDCalculator(wavelength="CuKa")
    - top 20 most intense peaks selected ($2\theta$ and int)
    - Normalisations
      - $2\theta$ min-max for 0,90
      - intensities min-max for 0,100
- Deduplicated
- Cleaned for CIF augmentation
  -  Note: I didnt filter to context length here because it was not implemented yet, but filter to context was flagged as True during model training which effectively does the same thing (less efficient)
- dataset pushed to HuggingFace as: c-bone/mattergen_XRD (90:10 train/valid sets)

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/mattergen_XRD-slider.jsonc'

#### 2nd pass finetune - CHILI-100K XRD
- Dataset Source: [CHILI-100K](https://github.com/UlrikFriisJensen/CHILI)
  - Columns: `Database`, `Material ID`, `Reduced Formula`, `CIF`
  - CIF source: generated structures by extracting structures from the dataset and then CifWriter with symprec 0.1
- Preprocessing and filtering
  - deduplicated the raw parquet
  - Added theoretical XRD peaks with [_calculate_theor_XRD.py](_utils/_preprocessing/_calculate_theor_XRD.py)
    - pmg - `XRDCalculator(wavelength="CuKa")`
    - top 20 most intense peaks selected ($2\theta$ and int)
    - Normalisations
      - $2\theta$ min-max for 0,90
      - intensities min-max for 0,100
  - Ran VUN metrics against `c-bone/lematerial_clean` and filtered to the 1536-token context limit
  - Built a leakproof split with mutually exclusive novelty tiers
    - test: 500 seen + 500 structurally novel + 500 compositionally novel
    - val: 1500 in-distribution structures
    - train: remaining rows
- Final dataset cleaned for CIF augmentation and pushed to HuggingFace as: `c-bone/chili100k_strat`

#### Making the dataset

Full preprocessing pipeline: load raw -> deduplicate -> add theoretical XRD peaks -> tag novelty -> filter to context length -> build leakproof split -> clean -> push to HF.

**Step 1: have a look at the raw parquet**

In [ ]:
df = pd.read_parquet('/home/cyprien/Data_Gen/chili100k_data.parquet')
df.head()

**Step 2: Deduplicate** removes identical CIFs so they can't appear in both train and test

In [ ]:
!python _utils/_preprocessing/_deduplicate.py \
  --input_parquet /home/cyprien/Data_Gen/chili100k_data.parquet \
  --output_parquet HF-databases/chili100k/chili100k_dedup.parquet

**Step 3: Add theoretical XRD condition vectors** simulates Cu Kα XRD, selects the 20 most intense peaks, and normalises $2\theta$ and intensities to fixed ranges. This is the condition signal fed to the model.

In [ ]:
!python _utils/_preprocessing/_calculate_theor_XRD.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --output_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --num_workers 16

**Step 4: Tag novelty against the training database** each structure is labelled as *seen* (matches something in LeMaterial), *structurally novel* (same composition, different structure), or *compositionally novel* (entirely new composition). This is what lets us build the leakproof split and report stratified results later.

In [ ]:
!python _utils/_metrics/VUN_metrics.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup.parquet \
    --huggingface_dataset "c-bone/lematerial_clean" \
    --load_processed_data "HF-databases/lematerial/lematerial_dedup.parquet" \
    --output_parquet HF-databases/chili100k/chili100k_dedup_vun.parquet \
    --output_csv HF-databases/chili100k/chili100k_dedup_vun.csv \
    --check_comp_novelty \
    --num_workers 32 \
    --bond_length_acceptability_cutoff 0.0

> Note: We remove invalid CIFs here, but set bond_length_acceptability_cutoff to 0.0 to avoid filtering out the experimentally observed materials that would fail this check.
> Otherwise, the goal here is to make sure no CIF info is leaked from any training phase to test phase apart from the 500 seen subset

**Step 5: Filter to context length** count tokens per CIF and drop anything above 1536 (which will bethe model's context window). The two cells below do this: first count, then filter. For more stats on the dataset see [Y_Dataset_stats.ipynb](notebooks/Y_Dataset_stats.ipynb).

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup_vun.parquet \
    --output_parquet HF-databases/chili100k/chili100k_count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
df_count = pd.read_parquet('HF-databases/chili100k/chili100k_count.parquet')
df_dedup = pd.read_parquet('HF-databases/chili100k/chili100k_dedup_vun.parquet')

df_merged = pd.merge(df_dedup, df_count[['Material ID', 'token_count']], on='Material ID', how='inner')
df_filtered = df_merged[df_merged['token_count'] <= 1536]
df_filtered.to_parquet('HF-databases/chili100k/chili100k_dedup_vun_filtered.parquet', index=False)

**Step 6: Build the leakproof stratified split** from the tiered dataset, 500 samples are drawn from each novelty tier (seen, structurally novel, compositionally novel) for the test set. Val gets 1500 in-distribution structures. Everything else is train.

In [ ]:
df = pd.read_parquet('HF-databases/chili100k/chili100k_dedup_vun_filtered.parquet')

df = df[~((df['is_valid'] == False) & (df['is_unique'] == False))]
df = df.drop(columns=['is_valid', 'is_unique'])

# Define strictly mutually exclusive masks to prevent data overlap
# mask_comp_novel = df['is_comp_novel'] == True and 'Reduced Formula' is unique in the dataset
# We want to ensure that if a composition is marked as novel, it does not have duplicates in the dataset, which could leak information across splits. Therefore, we check for both conditions: is_comp_novel is True and the Reduced Formula is not duplicated (i.e., unique).
mask_comp_novel = (df['is_comp_novel'] == True) & (df['Reduced Formula'].duplicated(keep=False) == False)
print(f"Number of comp novel structures: {mask_comp_novel.sum()}")
mask_struct_novel = (df['is_novel'] == True) & (df['is_comp_novel'] == False)
mask_in_dist = df['is_novel'] == False 

# Sample the test set
test_comp = df[mask_comp_novel].sample(n=500, random_state=1)
test_struct = df[mask_struct_novel].sample(n=500, random_state=1)
test_in_dist = df[mask_in_dist].sample(n=500, random_state=1)
test_indices = pd.concat([test_comp, test_struct, test_in_dist]).index

# Filter out the test set to define the remaining pool
remaining_df = df.drop(test_indices)

# Sample the validation set (1500 from in-distribution structures)
val_indices = remaining_df[remaining_df['is_novel'] == False].sample(n=1500, random_state=1).index

df['Split'] = 'train'
df.loc[val_indices, 'Split'] = 'val'
df.loc[test_indices, 'Split'] = 'test'

print(df['Split'].value_counts())

# renam 'Generated CIF' to CIF
df = df.rename(columns={'Generated CIF': 'CIF'})
df.to_parquet('HF-databases/chili100k/chili100k_dedup_leakproof.parquet', index=False)

**Steps 7 & 8: Clean for CIF augmentation and push to HuggingFace** final formatting pass then upload as `c-bone/chili100k_strat`

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet 'HF-databases/chili100k/chili100k_clean.parquet' \
    --num_workers 32 \
    --count_tokens

In [ ]:
!python _utils/_preprocessing/_save_dataset_to_HF.py \
    --input_parquet 'HF-databases/chili100k/chili100k_clean.parquet' \
    --output_parquet 'HF-databases/chili100k_strat.parquet' \
    --save_hub

### Training

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/chili100k-xrd-slider-opt.jsonc'

> This model's weights are loaded from the 1st pass (Mattergen XRD) described at the top of the notebook

### Generating

First create prompts from the held-out CHILI-100K test split, then run repeated generations for the conditioned and text-only baselines.

Each prompt packages the formula header + XRD condition vector for one test structure — the model then completes the CIF. `level_3` uses the full composition + XRD signal; `condition_vector` is the encoded XRD peaks.

In [ ]:
!python _utils/_generating/make_prompts.py \
    --HF_dataset 'c-bone/chili100k_strat' \
    --split 'test' \
    --automatic \
    --output_parquet '_artifacts/chili-xrd/chili-test_prompts.parquet' \
    --level 'level_3' \
    --condition_columns 'condition_vector'

#### Generate materials from the Chili model and XRD information (Repeated 3x)

In [ ]:
!python _utils/_generating/generate_CIFs.py --config '_config_files/generation/conditional/xrd_studies/chili-xrd_eval.jsonc'

In [ ]:
!python _utils/_generating/postprocess.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_gen.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_post.parquet' \
    --num_workers 32 \
    --column_name 'Generated CIF'

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_gen.parquet' \
    --num_gens 20 \
    --ref_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_metrics.parquet' \
    --num_workers 32 \
    --validity_check "none"

`num_gens=20` uses all 20 generated structures per prompt (best-of-20 scoring). `num_gens=1` uses only the first perplexity ranked structure. Both are useful: 20-perp shows the ceiling of what the model can do, 1-perp shows what you'd get in a closed loop best guess

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-20perp-test_gen.parquet' \
    --num_gens 1 \
    --ref_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-1perp-test_metrics.parquet' \
    --num_workers 32 \
    --validity_check "none"

#### Comparison to text-only model

We mirror the exact same training pipeline as above, however here we never let the model see XRD info in any of the training phase to understand the gains from XRD conditioning across the board

> **Note**: This comparison keeps the same held-out CHILI-100K prompt split, but switches to the unconditional 2-pass finetuning so the effect of the XRD channel can be isolated.

**Pass 1: text-only Mattergen finetune** (same data as conditioned pass 1 but XRD stripped)

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/mattergen_XRD-uncond.jsonc'

**Pass 2: text-only CHILI-100K finetune**

In [ ]:
!torchrun --nproc_per_node=2 _train.py --config '_config_files/training/conditional/xrd_studies/chili100k-uncond-opt.jsonc'

Generate, postprocess, and score same sequence as the conditioned model above

In [ ]:
!python _utils/_generating/generate_CIFs.py --config '_config_files/generation/conditional/xrd_studies/chili-xrd-uncond_eval.jsonc'

In [ ]:
!python _utils/_generating/postprocess.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_gen.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_post.parquet' \
    --num_workers 32 \
    --column_name 'Generated CIF'

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_gen.parquet' \
    --num_gens 20 \
    --ref_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics.parquet' \
    --num_workers 32 \
    --validity_check "none"

In [ ]:
!python _utils/_metrics/XRD_metrics.py \
    --input_parquet '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_gen.parquet' \
    --num_gens 1 \
    --ref_parquet 'HF-databases/chili100k/chili100k_dedup_leakproof.parquet' \
    --output_parquet '_artifacts/chili-xrd/chili-ft-uncond-1perp-test_metrics.parquet' \
    --num_workers 32 \
    --validity_check "none"

Same scoring as above: `num_gens=20` for best-of-20, `num_gens=1` for single-shot.

### Aggregate Results

Aggregate the repeated conditioned and text-only runs, then report stratified averages with standard errors across the three repeats.

Each model variant was run 3 times (v1/v2/v3) to get a sense of variance. Keys follow the pattern `<cond|uncond>-<Nperp>`: `cond` = XRD-conditioned, `uncond` = text-only baseline, `Nperp` = number of generations used for scoring (20 = best-of-20, 1 = best ranked).

**Tiers:** *Memorized* = test structures the model has seen composition+structure of before; *Structurally Novel* = seen composition, new structure; *Compositionally Novel* = completely new composition. This lets us see where the XRD signal helps most.

In [ ]:
import __init__
import pandas as pd
import numpy as np
from _utils._notebook_utils.x_xrd_chili100k_utils import get_stratified_metrics_xrd

In [ ]:
paths = {
    'cond-20perp': '_artifacts/chili-xrd/chili-ft-20perp-test_metrics1.parquet',
    'cond-20perp-v2': '_artifacts/chili-xrd/chili-ft-20perp-test_metrics2.parquet',
    'cond-20perp-v3': '_artifacts/chili-xrd/chili-ft-20perp-test_metrics3.parquet',
    'cond-1perp': '_artifacts/chili-xrd/chili-ft-1perp-test_metrics1.parquet',
    'cond-1perp-v2': '_artifacts/chili-xrd/chili-ft-1perp-test_metrics2.parquet',
    'cond-1perp-v3': '_artifacts/chili-xrd/chili-ft-1perp-test_metrics3.parquet',
    'uncond-20perp': '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics1.parquet',
    'uncond-20perp-v2': '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics2.parquet',
    'uncond-20perp-v3': '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics3.parquet',
    'uncond-1perp': '_artifacts/chili-xrd/chili-ft-uncond-1perp-test_metrics1.parquet',
    'uncond-1perp-v2': '_artifacts/chili-xrd/chili-ft-uncond-1perp-test_metrics2.parquet',
    'uncond-1perp-v3': '_artifacts/chili-xrd/chili-ft-uncond-1perp-test_metrics3.parquet',
}

tier_sizes = {
    'Overall': 1500,
    'Memorized (Seen Comp & Struct)': 500,
    'Structurally Novel (Seen Comp)': 500,
    'Compositionally Novel (Unseen Comp)': 500,
}

tables = {}
for run_name, parquet_path in paths.items():
    df_metrics = pd.read_parquet(parquet_path)
    tables[run_name] = get_stratified_metrics_xrd(
        df_metrics,
        n_test=tier_sizes,
        only_matched=False,
        verbose=False,
    )

final_table = pd.concat(tables, names=['run', 'tier'])
final_table.index = [f'{run}__{tier}' for run, tier in final_table.index]
final_table.to_parquet('_artifacts/chili-xrd/chili-slider-vs-uncond-all-results.parquet')
final_table

In [ ]:

final_table = pd.read_parquet('_artifacts/chili-xrd/chili-slider-vs-uncond-all-results.parquet')

base_conditions = {}
for index in final_table.index:
    if '-v2' in index:
        base_name = index.replace('-v2', '')
    elif '-v3' in index:
        base_name = index.replace('-v3', '')
    else:
        base_name = index
    
    if base_name not in base_conditions:
        base_conditions[base_name] = []
    base_conditions[base_name].append(index)

# averaged results with standard error between the 3 runs
averaged_results = {}
stderr_results = {}
for base_name, variants in base_conditions.items():
    variant_data = []
    for variant in variants:
        if variant in final_table.index:
            variant_data.append(final_table.loc[variant])
    
    if variant_data:
        data_df = pd.concat(variant_data, axis=1)
        averaged_results[base_name] = data_df.mean(axis=1)
        stderr_results[base_name] = data_df.std(axis=1) / np.sqrt(len(variant_data))

averaged_table = pd.DataFrame.from_dict(averaged_results, orient='index')
stderr_table = pd.DataFrame.from_dict(stderr_results, orient='index')

_ATOM_COUNT_COLS = {col for col in averaged_table.columns if 'Atom Count' in col}

formatted_table = pd.DataFrame(index=averaged_table.index, columns=averaged_table.columns, dtype=object)
for col in averaged_table.columns:
    fmt = ".0f" if col in _ATOM_COUNT_COLS else ".3f"
    for idx in averaged_table.index:
        mean_val = averaged_table.loc[idx, col]
        stderr_val = stderr_table.loc[idx, col]
        if pd.isna(stderr_val) or stderr_val == 0:
            formatted_table.loc[idx, col] = f"{mean_val:{fmt}}"
        else:
            formatted_table.loc[idx, col] = f"{mean_val:{fmt}} (±{stderr_val:{fmt}})"

formatted_table.to_csv('plots/chili/chili-slider-vs-uncond-all-results_formatted.csv')
formatted_table


> Note: the results here differ very slightly from any pooled plots because this table averages each repeated run first and then reports the associated stderr, rather than concatenating all rows across repeats before computing summary statistics.

### Plots

Scatter plots of true XRD peaks (from the reference structure) vs generated XRD peaks (from the best matched generated structure). Points on the diagonal mean perfect recovery. Matched structures (a best-of-20 match was found) are shown separately from unmatched ones. The conditioned model should cluster tighter around the diagonal than the text-only baseline.

In [ ]:
import __init__
from _utils._notebook_utils.x_xrd_chili100k_utils import load_and_process_xrd_metrics, plot_true_vs_gen

cond_files = [
    '_artifacts/chili-xrd/chili-ft-20perp-test_metrics1.parquet',
    '_artifacts/chili-xrd/chili-ft-20perp-test_metrics2.parquet',
    '_artifacts/chili-xrd/chili-ft-20perp-test_metrics3.parquet'
]

df_metrics = load_and_process_xrd_metrics(cond_files)

plot_true_vs_gen(
    df_metrics,
    figsize=(14, 14),
    title_fontsize=26,
    label_fontsize=24,
    ticks_fontsize=22,
    legend_fontsize=18,
    annot_fontsize=22,
    scatter_edge_width=0.5,
    alpha_matched=0.6,
    alpha_unmatched=0.6,
    min_marker_size=3,
    size_multiplier=6.0,
    diag_line_width=1.0,
    axes_linewidth=2.0,
    show_match_legend=True,
    show_size_legend=False,
    savepath="plots/chili/chili_xrd_cond.png"
)

Same plot for the text-only baseline — compare directly to the conditioned plot above to see the effect of the XRD signal.

In [ ]:
uncond_files = [
    '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics1.parquet',
    '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics2.parquet',
    '_artifacts/chili-xrd/chili-ft-uncond-20perp-test_metrics3.parquet'
]

df_metrics_uncond = load_and_process_xrd_metrics(uncond_files)

plot_true_vs_gen(
    df_metrics_uncond,
    figsize=(14, 14),
    title_fontsize=26,
    label_fontsize=24,
    ticks_fontsize=22,
    legend_fontsize=18,
    annot_fontsize=22,
    scatter_edge_width=0.5,
    alpha_matched=0.6,
    alpha_unmatched=0.6,
    min_marker_size=3,
    size_multiplier=6.0,
    diag_line_width=1.0,
    axes_linewidth=2.0,
    show_match_legend=False,
    show_size_legend=True,
    savepath="plots/chili/chili_xrd_uncond.png"
)

In [ ]:
df_metrics